In [0]:

CREATE VOLUME dev.bronze.landing
COMMENT 'this is landing managed volume';

In [0]:
%python
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/input")

True

In [0]:
%python
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-01.csv", "/Volumes/dev/bronze/landing/input")

True

In [0]:
%python
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-03.csv", "/Volumes/dev/bronze/landing/input")

True

In [0]:
CREATE TABLE dev.bronze.invoice_cp;

In [0]:
COPY INTO dev.bronze.invoice_cp
FROM '/Volumes/dev/bronze/landing/input'
FILEFORMAT = CSV
PATTERN = '*.csv'
FORMAT_OPTIONS(
    "header" = "true",
    "mergeSchema" = "true"
)
COPY_OPTIONS (
    "mergeSchema" = "true"
)
;

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


In [0]:
select * from dev.bronze.invoice_cp limit 50;

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


In [0]:
CREATE TABLE dev.bronze.invoice_cp_alt (
    InvoiceNo string,
    StockCode string,
    Quantity double,
    _insert_date timestamp
)
;

In [0]:
COPY INTO dev.bronze.invoice_cp_alt
FROM (SELECT InvoiceNo, StockCode, cast(Quantity as double)Quantity, current_timestamp() as _insert_date FROM
    "/Volumes/dev/bronze/landing/input"
    )
FILEFORMAT = CSV
PATTERN = '*.csv'
FORMAT_OPTIONS(
    "header" = "true",
    "mergeSchema" = "true"
)
;

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
2202,2202,0


In [0]:
select * from dev.bronze.invoice_cp_alt limit 50;

InvoiceNo,StockCode,Quantity,_insert_date
536365,85123A,6.0,2026-10-15T23:02:05.396Z
536365,71053,6.0,2026-10-15T23:02:05.396Z
536365,84406B,8.0,2026-10-15T23:02:05.396Z
536365,84029G,6.0,2026-10-15T23:02:05.396Z
536365,84029E,6.0,2026-10-15T23:02:05.396Z
536365,22752,2.0,2026-10-15T23:02:05.396Z
536365,21730,6.0,2026-10-15T23:02:05.396Z
536366,22633,6.0,2026-10-15T23:02:05.396Z
536366,22632,6.0,2026-10-15T23:02:05.396Z
536367,84879,32.0,2026-10-15T23:02:05.396Z


In [0]:
SELECT COUNT(1) FROM dev.bronze.invoice_cp_alt;

COUNT(1)
7419
